In [1]:
import pandas as pd

data = pd.read_csv('train_1m.txt', header = None, delimiter="\t")

In [2]:
from nn_model import DeepFM
from dataset import CustomDataset

#### Train

In [17]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from utils import save_meta_data

### Hyper Parameter
device = "cpu"
valid_ratio = 0.1
batch_size = 2000
init_lr = 1e-1
min_lr = 1e-2
save_path = "./saved_model"
total_epoch = 5
overfitting_limit = 5
random_seed = 999


## Split Dataset to Train & Valid
valid_data_1 = data[data[0] == 0].sample(frac = valid_ratio, random_state=random_seed)
valid_data_2 = data[data[0] == 1].sample(frac = valid_ratio, random_state=random_seed)
valid_df = pd.concat([valid_data_1, valid_data_2])
valid_df = valid_df.sample(frac=1)

valid_indices = set(valid_df.index)

train_df = data[~data.index.isin(valid_indices)]

## Train Dataset
train_dataset = CustomDataset(
    df = train_df,
    dataset_type = "train"
)
train_loader = DataLoader(
    train_dataset, 
    batch_size = batch_size, 
    shuffle=True, 2
    num_workers=0
)
categorical_layers, catalog_nums, numerical_meta = train_dataset.export_data()

## Save Meta Data
save_meta_data(
    categorical_layers = categorical_layers,
    catalog_nums = catalog_nums,
    numerical_meta = numerical_meta,
    save_path = save_path,
)

## Valid Dataset
valid_dataset = CustomDataset(
    df = valid_df,
    dataset_type = "valid",
    categorical_layers = categorical_layers,
    catalog_nums = catalog_nums,
    numerical_meta = numerical_meta
)
valid_loader = DataLoader(
    valid_dataset, 
    batch_size = batch_size, 
    shuffle=False,
    num_workers=0
)

## Model
model = DeepFM(
    catalog_nums=catalog_nums,
    numerical_len=13,
    categorical_len=26,
    embedding_dim=4,
    hidden_dim=4,
    bucket_nums=4,
    hidden_layer_nums=4,
    dropout_ratio=0.3
)
model.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = init_lr, 
    betas=(0.99, 0.999),
    weight_decay = 1e-5,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max= 5, 
    eta_min = min_lr
)
criterion = nn.BCELoss().to(device)

In [26]:
import torch.nn as nn
loss = nn.BCELoss()

BCELoss()

#### Model Loop

In [23]:

### Train
import time
import os
import sys

import numpy as np

from utils import calculate_confusion_mat, calculate_score
 

def model_loop(
    model,
    data_loader,
    is_train: bool = True,
    device: str = "cpu",
    logging_nums: int = 10
):  
    if is_train:
        model.train()
    else:
        model.eval()

    step = 1
    steps = len(data_loader)
    logging_step_size = steps // logging_nums
    
    ## loss
    total_loss = []
    temp_loss = []
    
    ## pred, label
    total_logits = []
    total_labels = []
    
    ## score
    total_cm = {
        "tp": 0, "fp": 0, "fn": 0, "tn": 0
    }
    
    prev_time = time.time()
    for data in iter(data_loader):
        label, numeric_values, categoric_values, numeric_indicators, categoric_indicators = data

        label = label.float().to(device)
        numeric_values = numeric_values.to(device)
        categoric_values = categoric_values.to(device)
        numeric_indicators = numeric_indicators.to(device)
        categoric_indicators = categoric_indicators.to(device)
        
        if is_train:
            optimizer.zero_grad()

        logit = model(
            numeric_values,
            categoric_values,
            numeric_indicators,
            categoric_indicators
        )                
        
        loss = criterion(logit.squeeze(1), label)
        
        if is_train:
            loss.backward()
            optimizer.step()

        ## Calculate Loss, Score
        temp_loss.append(loss.item())
        total_loss.append(loss.item())
        
        ## Score
        logits = list(logit.cpu().squeeze(1).detach().numpy())
        labels = list(label.cpu().detach().numpy())
        
        total_logits.extend(logits)
        total_labels.extend(labels)
        
        temp_cm = calculate_confusion_mat(
            logits=logits,
            labels=labels,
            theshold=0.5
        )
        for _k, _v in temp_cm.items():
            total_cm[_k] += _v
        
        ## Logging
        if step % logging_step_size == 0:
            current_loss = np.mean(temp_loss)
            timedelta = time.time() - prev_time
            accuracy, precision, recall, f1 = calculate_score(
                total_cm=temp_cm
            )
            
            print(
                f"[ Step : ({step}/{steps}) "
                f"| Loss : {current_loss:.4f} "
                f"| Time : {timedelta:.4f}s]"
            )
            print(
                f"[ Accuracy : {accuracy:.6f} "
                f"| Precision : {precision:.6f} "
                f"| Recall : {recall:.6f} "
                f"| F1 : {f1:.6f}]"
            )
            sys.stdout.flush()
            
            temp_loss.clear()
            prev_time = time.time()

        step = step+1
    
    ## For epoch summary
    _total_loss = np.mean(total_loss)
    total_accuracy, total_precision, total_recall, total_f1 = calculate_score(
        total_cm=total_cm
    )
    total_auc = roc_auc_score(total_labels, total_logits)
    print("[ Total Result ]")
    print(
        f"[ Loss : {_total_loss:.4f} "
        f"| Accuracy : {total_accuracy:.6f} "
        f"| Precision : {total_precision:.6f} "
        f"| Recall : {total_recall:.6f} "
        f"| F1 : {total_f1:.6f} "
        f"| AUC : {total_auc:.6f} ]"
    )
    
    return {
        "loss": _total_loss,
        "accuracy": total_accuracy,
        "precision": total_precision,
        "recall": total_recall,
        "f1": total_f1,
        "auc": total_auc,
        "cm": total_cm,
        "logits": total_logits,
        "labels": total_labels
    }


In [24]:
import os

best_loss = 1e3
not_updated = 0

total_train_outputs = []
total_valid_outputs = []

for epoch in range(1, total_epoch+1):
    print(f"###### Train : ({epoch}/{total_epoch}) ######")
    ## Train
    train_output = model_loop(
        model=model,
        data_loader=train_loader,
        is_train=True,
        device=device,
        logging_nums=5
    )
    total_train_outputs.append(train_output)
    
    print(f"###### Validation : ({epoch}/{total_epoch}) ######")
    ## Validation
    valid_output = model_loop(
        model=model,
        data_loader=valid_loader,
        is_train=False,
        device=device,
        logging_nums=2
    )
    total_valid_outputs.append(valid_output)
    
    # scheduler.step()

    ## Save
    if valid_output["loss"] < best_loss:
        not_updated = 0
        best_loss = valid_output["loss"]    
        if not os.path.isdir(save_path):
            os.mkdir(save_path)     

        full_save_path = os.path.join(save_path, "weight.pth")
        print("Saving Model..")
        torch.save(model, full_save_path)    
    else:
        not_updated += 1
    
    if not_updated >= overfitting_limit:
        print("End Training..")
        break


###### Train : (1/5) ######
[ Step : (90/450) | Loss : 17.6872 | Time : 20.5588s]
[ Accuracy : 0.581500 | Precision : 0.254286 | Recall : 0.361055 | F1 : 0.298407]
[ Step : (180/450) | Loss : 17.1329 | Time : 20.4193s]
[ Accuracy : 0.609000 | Precision : 0.257340 | Recall : 0.297405 | F1 : 0.275926]
[ Step : (270/450) | Loss : 17.4075 | Time : 20.3974s]
[ Accuracy : 0.610500 | Precision : 0.274168 | Recall : 0.350202 | F1 : 0.307556]
[ Step : (360/450) | Loss : 17.4544 | Time : 20.6133s]
[ Accuracy : 0.642000 | Precision : 0.293478 | Recall : 0.331967 | F1 : 0.311538]
[ Step : (450/450) | Loss : 17.5881 | Time : 19.2856s]
[ Accuracy : 0.611000 | Precision : 0.294599 | Recall : 0.341556 | F1 : 0.316344]
[ Total Result ]
[ Loss : 17.4540 | Accuracy : 0.610841 | Precision : 0.285237 | Recall : 0.349582 | F1 : 0.314149 | AUC : 0.569503 ]
###### Validation : (1/5) ######
[ Step : (25/50) | Loss : 16.8337 | Time : 4.7019s]
[ Accuracy : 0.606500 | Precision : 0.292570 | Recall : 0.364162 | F1

KeyboardInterrupt: 

In [22]:
from sklearn.metrics import roc_auc_score 

y_true = valid_output["labels"]
y_pred = valid_output["logits"]



0.5695034331247277

In [6]:
train_output.keys()

dict_keys(['loss', 'accuracy', 'precision', 'recall', 'f1', 'cm', 'logits', 'labels'])

In [8]:
len(valid_output["logits"])

100000

In [9]:
len(valid_output["labels"])

100000

In [ ]:
x_numeric = torch.tensor(
    [
        [0.1, 0.2, 0.3],
        [0.4, 0.5, 0.7]
    ],
)
x_numeric.to("cpu")

x_categoric = torch.tensor(
    [
        [1, 5],
        [10, 20]
    ],
)
x_categoric.to("cpu")

x_numeric_indicator = torch.tensor(
    [
        [1, 1, 0],
        [1, 1, 1]
    ]
)
x_numeric_indicator.to("cpu")

x_categoric_indicator = torch.tensor(
    [
        [1, 1],
        [1, 0]
    ]
)
x_categoric_indicator.to("cpu")

model(
    x_numeric, 
    x_categoric,
    x_numeric_indicator,
    x_categoric_indicator
)

In [ ]:
layer = nn.Linear(
    in_features=1, 
    out_features=4
)  
x_input = torch.tensor(
    [[1], [2], [3.0]]
)

layer(x_input)

In [ ]:

batch_size = 3
feature_nums = 4
embedding_dim = 2
hidden_dim = 4

x_input_embeddings = torch.tensor([
    [[0.1, 0.2], [0.9, 1.51], [0.4, 0.4], [0.5, 0.9]],
    [[-0.12, 10.0], [3.1, 0.4], [0.01, 0.3], [0.23, 0.5]],
    [[12.0, 2.0], [1.4, 4.3], [5.3, 5.2], [1.2, 4.2]]
])      ## batch size x feature nums x embedding dim  

x_indicator = torch.tensor([
    [1, 0, 1, 1],
    [1, 1, 1, 1],
    [1, 1, 1, 0]   
])       ## batch size x feature nums